# DQN Buy Allocator — Alpaca Paper

This notebook:
1) Runs **fresh DQN inference** for your symbols using `rl_pipeline3_rs_std_patched.py`.
2) Loads the latest `infer_*.csv`, filters to **BUY** rows and reads `pos_frac`.
3) Allocates **deployable cash** across BUYs (normalized `pos_frac`).
4) Places **fractional notional** BUY market orders (falls back to whole-share `qty` if non-fractionable).
5) Saves artifacts under `./alpaca_paper/out`.

Use at the open (RTH) for realistic fills. For extended-hours testing, switch to LIMIT + `extended_hours=True` (we can add a toggle later).

python .\alpaca_paper\intraday_allocator_from_infers.py `
  --strategy S8 `
  --infer-refresh-hours 2 `
  --once `
  --dry-run

In [113]:
from pathlib import Path

ALGO_ROOT = Path(r"C:\Users\brobi\OneDrive\Desktop\Algo1")
RL_SCRIPT = ALGO_ROOT / "rl_pipeline3_rs_std_patched.py"

print("ALGO_ROOT:", ALGO_ROOT)
print("RL_SCRIPT:", RL_SCRIPT)

ALGO_ROOT: C:\Users\brobi\OneDrive\Desktop\Algo1
RL_SCRIPT: C:\Users\brobi\OneDrive\Desktop\Algo1\rl_pipeline3_rs_std_patched.py


In [114]:
# === Cell 0 — Alpaca Paper keys (set here for paper; move to env for live) ===
APCA_KEY_ID     = "PKUNRTQLNIJ4FWITDRXUCGZTPT"  # <-- your PAPER Key ID (or leave blank to use env)
APCA_SECRET_KEY = "6fGdcfwCzp8vHUYXacaoVLpPsknY5n4d9ngukbqFdYpU"  # <-- your PAPER Secret Key (or leave blank to use env)


In [115]:
# === Cell 1 — Imports, paths, and basic config ===

from __future__ import annotations

import os
import sys
import subprocess
import shutil
import asyncio
from pathlib import Path
from datetime import datetime, time as dtime, timedelta
from dateutil import tz

# --- Root project path (adjust if needed) ---
ALGO_ROOT = Path(r"C:\Users\brobi\OneDrive\Desktop\Algo1")

# RL pipeline script
RL_SCRIPT = ALGO_ROOT / "rl_pipeline3_rs_std_patched.py"

# Where Alpaca-related stuff lives
ALPACA_DIR = ALGO_ROOT / "alpaca"

# Raw daily infer logs from rl_pipeline (infer_YYYY-MM-DD.csv will land here)
INFER_RAW_DIR = ALPACA_DIR / "intraday_raw"

# Snapshot copies for EACH intraday run (what you want to inspect)
INFER_SNAP_DIR = ALPACA_DIR / "intraday_infers"

INFER_RAW_DIR.mkdir(parents=True, exist_ok=True)
INFER_SNAP_DIR.mkdir(parents=True, exist_ok=True)

# How often we want to run the intraday inference (hours)
INFER_INTERVAL_HOURS = 1

# Timezone and RTH window (for future use; we can still use it now if you like)
TZ = tz.gettz("America/Los_Angeles")
MARKET_START_PT = dtime(6, 30)  # 9:30 ET
MARKET_END_PT   = dtime(13, 0)  # 16:00 ET

# Whether to only run during RTH (True = skip outside RTH)
RTH_ONLY = True

# Which symbols to infer on each run (fill this with your live list)
SYMBOLS = [
    "IBM","RGTI","QBTS","QUBT","IONQ","QS","AMD","SLDP","MSFT","CHGG","AI","NVDA","TSM","GOOGL",
    "PAYO","LCID","PLUG","BYND","TM","SPY","AMC","TDC","INFA","SNOW","PSTG","MDB","FSLR",
    "ENPH","SEDG","ARRY","NXT","ENVX","MVST","EOSE","FLNC","EVGO","ITRI","AMSC","POWI","VICR",
    "NVTS","CLNE","GEVO","MNTK","ELVA","XEL","AEP","RNW","INTC","ARQQ","MU","SMCI","TRV","PGR",
    "BHP","COST","MRK","NFLX","RMBS","ALB","VZ","AAPL","PG","ROP","KO","PEP","T","TMUS","CMCSA",
    "CCI","KR","MDLZ","GIS","MKC","NOK","AVGO","ASML","CSCO","AZN","SHOP","APP","LIN","LRCX",
    "QCOM","PDD","ISRG","INTU","ARM","AMAT","BKNG","KLAC","AMGN","TXN","PANW","ADBE","GILD",
    "CRWD","HON","CEG","ADI","MELI","ADP","DASH","VRTX","SBUX","CDNS","SNPS","MSTR","ORLY",
    "ABNB","MRVL","CTAS","MAR","TRI","PYPL","REGN","MNST","CSX","ADSK","FTNT","WDAY","AXON",
    "DDOG","NXPI","ROST","ZS","WBD","PCAR","IDXX","EA","EXC","FAST","BKR","TTWO","PAYX","TEAM",
    "CPRT","CCEP","FANG","KDP","GEHC","MCHP","CHTR","CTSH","VRSK","CSGP","KHC","ODFL","DXCM",
    "TTD","BIIB","LULU","ON","CDW","GFS","ATO","EVRG","WEC","COR","JNJ","TJX","CMS","PPL","DUK"
]

print("Symbols per run:", len(SYMBOLS))
print("RL script:", RL_SCRIPT)
print("Raw infer dir:", INFER_RAW_DIR)
print("Snapshot dir:", INFER_SNAP_DIR)


Symbols per run: 164
RL script: C:\Users\brobi\OneDrive\Desktop\Algo1\rl_pipeline3_rs_std_patched.py
Raw infer dir: C:\Users\brobi\OneDrive\Desktop\Algo1\alpaca\intraday_raw
Snapshot dir: C:\Users\brobi\OneDrive\Desktop\Algo1\alpaca\intraday_infers


In [116]:
# === Cell 2 — Helper to check Regular Trading Hours (RTH) ===

def is_rth(now: datetime | None = None) -> bool:
    """
    True if we are within regular trading hours (Mon–Fri, 9:30–16:00 ET).
    """
    if now is None:
        now = datetime.now(TZ)
    # Weekend guard
    if now.weekday() >= 5:  # 5=Sat, 6=Sun
        return False
    t = now.timetz().replace(tzinfo=None)
    return MARKET_START_PT <= t <= MARKET_END_PT

# Quick sanity check
now_test = datetime.now(TZ)
print("Now PT:", now_test)
print("Is RTH?", is_rth(now_test))


Now PT: 2025-11-25 05:37:31.859256-08:00
Is RTH? False


In [117]:
# === Cell 3 — Run rl_pipeline infer for ONE symbol (matching your daily CLI) ===
# === Updated Cell — Run rl_pipeline infer for ONE symbol using existing daily_cache ===

import sys
import os
import subprocess

def run_infer_for_symbol(symbol: str, log_dir: Path) -> None:
    """
    Run rl_pipeline3_rs_std_patched.py infer for a single symbol,
    using the existing daily_cache under ALGO_ROOT, and log to `log_dir`.

    This does:
      - ALPACA_FEED=iex
      - uses daily_cache/<SYMBOL>_daily.csv (no Alpaca refresh)
      - appends one row to infer_YYYY-MM-DD.csv in `log_dir`
    """
    env = dict(os.environ)
    env["ALPACA_FEED"] = "iex"  # same as your CLI

    cmd = [
        sys.executable,
        str(RL_SCRIPT),
        "infer",
        "--symbol", symbol,
        "--provisional-today",     # same-day decision
        "--log-csv", str(log_dir), # rl_pipeline will create infer_YYYY-MM-DD.csv here
        "--debug",
    ]

    print("[infer]", " ".join(cmd))
    # IMPORTANT: run from ALGO_ROOT so DAILY_CACHE_DIR = ALGO_ROOT/daily_cache
    result = subprocess.run(cmd, env=env, cwd=str(ALGO_ROOT))
    if result.returncode != 0:
        print(f"[infer] WARNING: infer failed for {symbol} (exit={result.returncode})")



In [118]:
# === Cell 4 — Run one intraday inference snapshot ===

def run_intraday_infer_snapshot() -> Path | None:
    """
    Run infer for all SYMBOLS, logging into INFER_RAW_DIR, then
    copy today's infer_YYYY-MM-DD.csv to a timestamped snapshot file in INFER_SNAP_DIR.

    Returns the snapshot path (or None if no raw file is found).
    """
    now = datetime.now(TZ)
    print(f"\n[snapshot] Starting intraday infer run at {now:%Y-%m-%d %H:%M:%S} PT")

    # 1) Run infer for each symbol
    for sym in SYMBOLS:
        run_infer_for_symbol(sym, INFER_RAW_DIR)

    # 2) Figure out today's raw infer file
    day_str = now.date().isoformat()  # YYYY-MM-DD
    raw_file = INFER_RAW_DIR / f"infer_{day_str}.csv"
    if not raw_file.exists():
        print(f"[snapshot] WARNING: raw infer file not found: {raw_file}")
        return None

    # 3) Copy to a timestamped snapshot
    ts = now.strftime("%Y%m%d_%H%M")
    snap_path = INFER_SNAP_DIR / f"infer_intraday_{ts}.csv"
    shutil.copy2(raw_file, snap_path)

    print(f"[snapshot] Saved snapshot to: {snap_path}")
    return snap_path


In [119]:
# === Cell 4b — One-off test run (no loop yet) ===

snap = run_intraday_infer_snapshot()
print("Snapshot:", snap)



[snapshot] Starting intraday infer run at 2025-11-25 05:37:31 PT
[infer] c:\Users\brobi\OneDrive\Desktop\Algo1\.venv\Scripts\python.exe C:\Users\brobi\OneDrive\Desktop\Algo1\rl_pipeline3_rs_std_patched.py infer --symbol IBM --provisional-today --log-csv C:\Users\brobi\OneDrive\Desktop\Algo1\alpaca\intraday_raw --debug
[infer] c:\Users\brobi\OneDrive\Desktop\Algo1\.venv\Scripts\python.exe C:\Users\brobi\OneDrive\Desktop\Algo1\rl_pipeline3_rs_std_patched.py infer --symbol RGTI --provisional-today --log-csv C:\Users\brobi\OneDrive\Desktop\Algo1\alpaca\intraday_raw --debug
[infer] c:\Users\brobi\OneDrive\Desktop\Algo1\.venv\Scripts\python.exe C:\Users\brobi\OneDrive\Desktop\Algo1\rl_pipeline3_rs_std_patched.py infer --symbol QBTS --provisional-today --log-csv C:\Users\brobi\OneDrive\Desktop\Algo1\alpaca\intraday_raw --debug
[infer] c:\Users\brobi\OneDrive\Desktop\Algo1\.venv\Scripts\python.exe C:\Users\brobi\OneDrive\Desktop\Algo1\rl_pipeline3_rs_std_patched.py infer --symbol QUBT --provi

In [120]:
import importlib
import strategies_s11_s13

print("Module file actually used:", strategies_s11_s13.__file__)

# Force Python to re-read the file from disk
strategies_s11_s13 = importlib.reload(strategies_s11_s13)

# Inspect what's in the module now
print("Has load_all_inf_from_logs?", hasattr(strategies_s11_s13, "load_all_inf_from_logs"))
print("Exports:", [n for n in dir(strategies_s11_s13) if "base" in n or "alloc" in n or "load_all" in n])


Module file actually used: c:\Users\brobi\OneDrive\Desktop\Algo1\alpaca_paper\strategies_s11_s13.py
Has load_all_inf_from_logs? True
Exports: ['build_base_and_today', 'build_s11_allocation', 'build_s11_allocation_for_signals', 'build_s11_live_allocation_from_snapshot', 'build_s12_allocation', 'build_s13_allocation', 'build_simple_allocation_from_signals', 'load_all_inf_from_logs']


In [121]:
import importlib, strategies_s11_s13
strategies_s11_s13 = importlib.reload(strategies_s11_s13)

from pathlib import Path
from strategies_s11_s13 import (
    load_inf_from_daily_and_intraday,
    build_base_and_today,
    build_s11_live_allocation_from_snapshot,
)

DAILY_LOGS_DIR    = Path(r"C:\Users\brobi\OneDrive\Desktop\Algo1\logs")
INTRADAY_LOGS_DIR = Path(r"C:\Users\brobi\OneDrive\Desktop\Algo1\alpaca\intraday_raw")

all_inf = load_inf_from_daily_and_intraday(DAILY_LOGS_DIR, INTRADAY_LOGS_DIR)
base, today_inf = build_base_and_today(all_inf)

alloc_s11_live = build_s11_live_allocation_from_snapshot(base, today_inf)
alloc_s11_live.head()


[load_all_inf] dir=C:\Users\brobi\OneDrive\Desktop\Algo1\logs pattern=infer_*.csv files=50 (daily)
[load_all_inf] daily: rows=2830, date_range=2025-09-15→2025-11-24
[load_all_inf] dir=C:\Users\brobi\OneDrive\Desktop\Algo1\alpaca\intraday_raw pattern=infer_*.csv files=3 (intraday)
[load_all_inf] intraday: rows=1790, date_range=2025-11-17→2025-11-24
[load_inf_combined] daily=2830, intraday=1790, combined=4620


,date,symbol,pct_S11_live


In [122]:
from strategies_s11_s13 import _prep_ev, _eligibility_columns

ev = _prep_ev(base)
ev = _eligibility_columns(ev)

# S11 state as of last hist row
last_state = (
    ev.sort_values(["symbol", "date"])
      .groupby("symbol", as_index=False)[["date", "elig_s11", "active_age_s11"]]
      .last()
      .set_index("symbol")[["elig_s11", "active_age_s11"]]
)

sig = today_inf.copy()
sig["symbol"] = sig["symbol"].astype(str)
sig = sig.join(last_state, on="symbol", how="left")
sig["elig_s11"] = sig["elig_s11"].fillna(False)

mask = (sig["side"].str.upper() == "BUY") & sig["elig_s11"]
print(sig.loc[mask, ["symbol", "date", "pos_frac", "elig_s11", "active_age_s11"]])

Empty DataFrame
Columns: [symbol, date, pos_frac, elig_s11, active_age_s11]
Index: []


In [123]:
import numpy as np
import pandas as pd

# Use the same knobs you have in strategies_s11_s13.py
S11_ACTIVATION_GOOD = 5
S11_DEACTIVATION_BAD = 3
S11_AGE_CAP = 60.0
S11_AGE_DIV = 10.0
EPS = 1e-9

def _prep_ev_inline(base_df: pd.DataFrame) -> pd.DataFrame:
    ev = base_df.copy()
    ev["date"] = pd.to_datetime(ev["date"])
    ev = ev.sort_values(["symbol", "date"]).reset_index(drop=True)

    ev["symbol"] = ev["symbol"].astype(str)
    ev["side"] = ev["side"].astype(str).str.upper()
    ev["is_buy"]  = ev["side"].eq("BUY")
    ev["is_hold"] = ev["side"].eq("HOLD")
    ev["is_sell"] = ev["side"].eq("SELL")

    if "pos_frac" not in ev.columns:
        ev["pos_frac"] = 1.0

    ev["score"] = ev["score"] if "score" in ev.columns else np.nan
    ev["n_seen_prior"] = ev.groupby("symbol").cumcount()
    return ev

def _elig_s11_inline(ev: pd.DataFrame) -> pd.DataFrame:
    ev = ev.sort_values(["symbol", "date"]).copy()
    ev["elig_s11"] = False
    ev["active_age_s11"] = 0.0

    for sym, df_sym in ev.groupby("symbol", sort=False):
        active = False
        good_streak = 0
        bad_streak = 0
        age = 0.0

        for idx, row in df_sym.iterrows():
            ev.at[idx, "elig_s11"] = active
            ev.at[idx, "active_age_s11"] = age

            is_signal = bool(row.get("is_buy", False)) or bool(row.get("is_sell", False))
            if not is_signal:
                if active:
                    age += 1.0
                continue

            is_corr = bool(row["correct"])
            if is_corr:
                good_streak += 1
                bad_streak = 0
                if (not active) and good_streak >= S11_ACTIVATION_GOOD:
                    active = True
                    age = 0.0
            else:
                bad_streak += 1
                good_streak = 0
                if active and bad_streak >= S11_DEACTIVATION_BAD:
                    active = False
                    age = 0.0

            if active:
                age += 1.0

    ev["elig_s11"] = ev["elig_s11"].fillna(False)
    ev["active_age_s11"] = ev["active_age_s11"].fillna(0.0)
    return ev

def s11_live_alloc_from_base_today(base: pd.DataFrame,
                                   today: pd.DataFrame) -> pd.DataFrame:
    """
    Compute S11 live allocation directly from base + today_inf.

    Returns DataFrame[date, symbol, pct_S11_live].
    """
    if base.empty or today.empty:
        return pd.DataFrame(columns=["date", "symbol", "pct_S11_live"])

    # --- 1) Historical S11 state from base ---
    ev = _prep_ev_inline(base)
    ev = _elig_s11_inline(ev)

    last_state = (
        ev.sort_values(["symbol", "date"])
          .groupby("symbol", as_index=False)[["date", "elig_s11", "active_age_s11"]]
          .last()
          .set_index("symbol")[["elig_s11", "active_age_s11"]]
    )

    # --- 2) Join onto today's latest infer per symbol ---
    sig = today.copy()
    sig["symbol"] = sig["symbol"].astype(str)
    sig["date"] = pd.to_datetime(sig["date"], errors="coerce")
    sig["side"] = sig["side"].astype(str).str.upper()
    if "pos_frac" not in sig.columns:
        sig["pos_frac"] = 1.0

    sig = sig.join(last_state, on="symbol", how="left")
    sig["elig_s11"] = sig["elig_s11"].fillna(False)
    sig["active_age_s11"] = pd.to_numeric(sig["active_age_s11"], errors="coerce").fillna(0.0)

    # --- 3) Filter to BUY & S11-active ---
    buys = sig[(sig["side"] == "BUY") & (sig["elig_s11"])].copy()
    print("S11 live debug:")
    print("  total today rows    :", len(sig))
    print("  total BUY rows      :", (sig["side"] == "BUY").sum())
    print("  BUY & S11-active rows:", len(buys))

    if buys.empty:
        return pd.DataFrame(columns=["date", "symbol", "pct_S11_live"])

    # --- 4) Weights: pos_frac * (1 + age / AGE_DIV), floor pos_frac <=0 to 1 ---
    base_w = pd.to_numeric(buys["pos_frac"], errors="coerce").fillna(0.0)
    base_w = np.where(base_w <= 0.0, 1.0, base_w)

    ages = np.clip(buys["active_age_s11"].to_numpy(dtype=float), 0.0, S11_AGE_CAP)
    mult = 1.0 + ages / S11_AGE_DIV

    w = base_w * mult
    weights = pd.Series(w, index=buys["symbol"]).groupby(level=0).sum()
    weights = weights[weights > 0]

    print("  symbols with positive weight:", len(weights))

    if weights.empty:
        return pd.DataFrame(columns=["date", "symbol", "pct_S11_live"])

    total = float(weights.sum())
    if not np.isfinite(total) or total <= 0:
        return pd.DataFrame(columns=["date", "symbol", "pct_S11_live"])

    pct = weights / total
    d = sig["date"].max()

    out = pd.DataFrame(
        {
            "date": pd.to_datetime(d).normalize(),
            "symbol": pct.index.astype(str),
            "pct_S11_live": pct.values,
        }
    ).sort_values("pct_S11_live", ascending=False).reset_index(drop=True)

    return out

# ---- run it on your current base + today_inf ----
alloc_s11_live = s11_live_alloc_from_base_today(base, today_inf)
alloc_s11_live.head()


S11 live debug:
  total today rows    : 161
  total BUY rows      : 61
  BUY & S11-active rows: 0


,date,symbol,pct_S11_live


In [124]:
# === Cell 5 — 2-hour intraday infer loop (files only, no trading) ===

async def intraday_infer_loop():
    print("✅ Intraday infer loop started "
          f"(interval={INFER_INTERVAL_HOURS}h, RTH_ONLY={RTH_ONLY})")

    while True:
        now = datetime.now(TZ)
        if RTH_ONLY and not is_rth(now):
            print(f"[loop] {now:%Y-%m-%d %H:%M:%S} PT — outside RTH, skipping run")
        else:
            try:
                run_intraday_infer_snapshot()
            except Exception as e:
                print("[loop] ERROR during run_intraday_infer_snapshot:", e)

        # Sleep for the configured interval
        delay = max(60.0, INFER_INTERVAL_HOURS * 3600.0)
        next_time = now + timedelta(seconds=delay)
        print(f"[loop] Next run around ~{next_time:%Y-%m-%d %H:%M:%S} PT "
              f"(sleeping {int(delay)}s)")
        await asyncio.sleep(delay)


In [ ]:
# === Cell 6 — Start the background loop ===

infer_task = asyncio.create_task(intraday_infer_loop())
print("intraday infer task created — leave this kernel running.")


intraday infer task created — leave this kernel running.


✅ Intraday infer loop started (interval=1h, RTH_ONLY=True)
[loop] 2025-11-25 05:47:29 PT — outside RTH, skipping run
[loop] Next run around ~2025-11-25 06:47:29 PT (sleeping 3600s)

[snapshot] Starting intraday infer run at 2025-11-25 06:47:29 PT
[infer] c:\Users\brobi\OneDrive\Desktop\Algo1\.venv\Scripts\python.exe C:\Users\brobi\OneDrive\Desktop\Algo1\rl_pipeline3_rs_std_patched.py infer --symbol IBM --provisional-today --log-csv C:\Users\brobi\OneDrive\Desktop\Algo1\alpaca\intraday_raw --debug
[infer] c:\Users\brobi\OneDrive\Desktop\Algo1\.venv\Scripts\python.exe C:\Users\brobi\OneDrive\Desktop\Algo1\rl_pipeline3_rs_std_patched.py infer --symbol RGTI --provisional-today --log-csv C:\Users\brobi\OneDrive\Desktop\Algo1\alpaca\intraday_raw --debug
[infer] c:\Users\brobi\OneDrive\Desktop\Algo1\.venv\Scripts\python.exe C:\Users\brobi\OneDrive\Desktop\Algo1\rl_pipeline3_rs_std_patched.py infer --symbol QBTS --provisional-today --log-csv C:\Users\brobi\OneDrive\Desktop\Algo1\alpaca\intrada

In [126]:
# === Cell 7 — Stop the background loop ===

infer_task.cancel()
await asyncio.gather(infer_task, return_exceptions=True)
print("intraday infer loop stopped ✅")


intraday infer loop stopped ✅
